In [1]:
%run ../setup_data.ipynb

No active 'spark' session found to stop.
Starting new Spark session: 'working_with_different_data_type'...


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/suanki/Desktop/pyspark-env/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/11 18:22:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


flight path verified (6 items): /home/suanki/Desktop/projects/suanki-pyspark-and-optimization/data/flight-data/csv
retail path verified (7 items): /home/suanki/Desktop/projects/suanki-pyspark-and-optimization/data/retail-data
Loading flight data...
Loading retail data...
+---------+-------------+-----------+
|namespace|    tableName|isTemporary|
+---------+-------------+-----------+
|sujeet_db|   department|      false|
|sujeet_db|       flight|      false|
|sujeet_db|      payment|      false|
|sujeet_db|       retail|      false|
|sujeet_db|sales_metrics|      false|
|sujeet_db| skew_metrics|      false|
|sujeet_db|weather_sales|      false|
+---------+-------------+-----------+

@@@@@@@@@@@@@@@@@
creating dataset 'emp_df' and 'dept_df' for join playbook
@@@@@@@@@@@@@@@@@@
root
 |-- emp_id: integer (nullable = true)
 |-- emp_name: string (nullable = true)
 |-- dept_id: integer (nullable = true)
 |-- details: struct (nullable = true)
 |    |-- location: string (nullable = true)
 |    

### Join Expressions
A join brings together two sets of data, the left and the right, by comparing the value of one or
more keys of the left and right and evaluating the result of a join expression that determines
whether Spark should bring together the left set of data with the right set of data.

## Join Types
Whereas the join expression determines whether two rows should join,
the join type determines what should be in the result set. 

There are a variety of different join types available in Spark for you to use:

- Inner joins (keep rows with keys that exist in the left and right datasets)

- Outer joins (keep rows with keys in either the left or right datasets)

- Left outer joins (keep rows with keys in the left dataset)

- Right outer joins (keep rows with keys in the right dataset)

- Left semi joins (keep the rows in the left, and only the left, dataset where the key
appears in the right dataset)

- Left anti joins (keep the rows in the left, and only the left, dataset where they do not
appear in the right dataset)

- Natural joins (perform a join by implicitly matching the columns between the two
datasets with the same names)

- Cross (or Cartesian) joins (match every row in the left dataset with every row in the
right dataset)

In [2]:
dept_df.show()

+-------+----------------+--------------------+--------------------+
|dept_id|       dept_name|             hq_info|          req_skills|
+-------+----------------+--------------------+--------------------+
|    101|Data Engineering|     {Pune, Tower A}|[Spark, Scala, Ka...|
|    102|       Analytics|{Bengaluru, Tower B}|[Python, SQL, Tab...|
|    103|       Cloud Ops|{Hyderabad, Tower C}|[AWS, Terraform, ...|
|    104|        Security|   {Mumbai, Tower A}| [Linux, Networking]|
+-------+----------------+--------------------+--------------------+



In [3]:
emp_df.show()

+------+--------+-------+---------------+-------------------+
|emp_id|emp_name|dept_id|        details|             skills|
+------+--------+-------+---------------+-------------------+
|     1|    Amit|    101|     {Pune, L3}|[Spark, Scala, AWS]|
|     2|   Priya|    102|{Bengaluru, L2}|      [Python, SQL]|
|     3|   Rohan|    101|   {Mumbai, L1}|      [Java, Spark]|
|     4|   Sneha|    105|     {Pune, L2}| [PySpark, Airflow]|
|     5|  Vikram|   NULL|    {Delhi, L4}|     [Kafka, Scala]|
+------+--------+-------+---------------+-------------------+



In [4]:
## Inner join

emp_df.join(dept_df, emp_df.dept_id == dept_df.dept_id, "inner").show(truncate=False)

+------+--------+-------+---------------+-------------------+-------+----------------+--------------------+----------------------+
|emp_id|emp_name|dept_id|details        |skills             |dept_id|dept_name       |hq_info             |req_skills            |
+------+--------+-------+---------------+-------------------+-------+----------------+--------------------+----------------------+
|1     |Amit    |101    |{Pune, L3}     |[Spark, Scala, AWS]|101    |Data Engineering|{Pune, Tower A}     |[Spark, Scala, Kafka] |
|3     |Rohan   |101    |{Mumbai, L1}   |[Java, Spark]      |101    |Data Engineering|{Pune, Tower A}     |[Spark, Scala, Kafka] |
|2     |Priya   |102    |{Bengaluru, L2}|[Python, SQL]      |102    |Analytics       |{Bengaluru, Tower B}|[Python, SQL, Tableau]|
+------+--------+-------+---------------+-------------------+-------+----------------+--------------------+----------------------+



In [5]:
## Full join/ Full outer join
## Keeps all rows from both datasets, filling missing sides with NULL.

emp_df.join(dept_df, emp_df.dept_id == dept_df.dept_id, "full").show(truncate=False)
emp_df.join(dept_df, emp_df.dept_id == dept_df.dept_id, "outer").show(truncate=False)

+------+--------+-------+---------------+-------------------+-------+----------------+--------------------+------------------------+
|emp_id|emp_name|dept_id|details        |skills             |dept_id|dept_name       |hq_info             |req_skills              |
+------+--------+-------+---------------+-------------------+-------+----------------+--------------------+------------------------+
|5     |Vikram  |NULL   |{Delhi, L4}    |[Kafka, Scala]     |NULL   |NULL            |NULL                |NULL                    |
|3     |Rohan   |101    |{Mumbai, L1}   |[Java, Spark]      |101    |Data Engineering|{Pune, Tower A}     |[Spark, Scala, Kafka]   |
|1     |Amit    |101    |{Pune, L3}     |[Spark, Scala, AWS]|101    |Data Engineering|{Pune, Tower A}     |[Spark, Scala, Kafka]   |
|2     |Priya   |102    |{Bengaluru, L2}|[Python, SQL]      |102    |Analytics       |{Bengaluru, Tower B}|[Python, SQL, Tableau]  |
|NULL  |NULL    |NULL   |NULL           |NULL               |103    |

In [6]:
### leftouter / left  join
# keeps all 5 employees from emp_df. Unmatched employees (Sneha, Vikram) get NULL for department columns.

emp_df.join(dept_df, emp_df.dept_id == dept_df.dept_id, "left").show(truncate=False)

+------+--------+-------+---------------+-------------------+-------+----------------+--------------------+----------------------+
|emp_id|emp_name|dept_id|details        |skills             |dept_id|dept_name       |hq_info             |req_skills            |
+------+--------+-------+---------------+-------------------+-------+----------------+--------------------+----------------------+
|1     |Amit    |101    |{Pune, L3}     |[Spark, Scala, AWS]|101    |Data Engineering|{Pune, Tower A}     |[Spark, Scala, Kafka] |
|2     |Priya   |102    |{Bengaluru, L2}|[Python, SQL]      |102    |Analytics       |{Bengaluru, Tower B}|[Python, SQL, Tableau]|
|3     |Rohan   |101    |{Mumbai, L1}   |[Java, Spark]      |101    |Data Engineering|{Pune, Tower A}     |[Spark, Scala, Kafka] |
|4     |Sneha   |105    |{Pune, L2}     |[PySpark, Airflow] |NULL   |NULL            |NULL                |NULL                  |
|5     |Vikram  |NULL   |{Delhi, L4}    |[Kafka, Scala]     |NULL   |NULL          

In [7]:
# Right Outer Join ("right", "rightouter")
# Keeps all 4 departments from dept_df. Unmatched departments (103, 104) get NULL for employee columns.


emp_df.join(dept_df, emp_df.dept_id == dept_df.dept_id, "right").show(truncate=False)

+------+--------+-------+---------------+-------------------+-------+----------------+--------------------+------------------------+
|emp_id|emp_name|dept_id|details        |skills             |dept_id|dept_name       |hq_info             |req_skills              |
+------+--------+-------+---------------+-------------------+-------+----------------+--------------------+------------------------+
|3     |Rohan   |101    |{Mumbai, L1}   |[Java, Spark]      |101    |Data Engineering|{Pune, Tower A}     |[Spark, Scala, Kafka]   |
|1     |Amit    |101    |{Pune, L3}     |[Spark, Scala, AWS]|101    |Data Engineering|{Pune, Tower A}     |[Spark, Scala, Kafka]   |
|2     |Priya   |102    |{Bengaluru, L2}|[Python, SQL]      |102    |Analytics       |{Bengaluru, Tower B}|[Python, SQL, Tableau]  |
|NULL  |NULL    |NULL   |NULL           |NULL               |103    |Cloud Ops       |{Hyderabad, Tower C}|[AWS, Terraform, Docker]|
|NULL  |NULL    |NULL   |NULL           |NULL               |104    |

In [8]:
## Left Semi Join ("left_semi", "semi")
## Works like a filter (WHERE EXISTS). Returns only columns 
# from emp_df where dept_id has a match in dept_df. You cannot select columns from dept_df.
emp_df.join(dept_df, emp_df.dept_id == dept_df.dept_id, "left_semi").show(truncate=False)

+------+--------+-------+---------------+-------------------+
|emp_id|emp_name|dept_id|details        |skills             |
+------+--------+-------+---------------+-------------------+
|1     |Amit    |101    |{Pune, L3}     |[Spark, Scala, AWS]|
|3     |Rohan   |101    |{Mumbai, L1}   |[Java, Spark]      |
|2     |Priya   |102    |{Bengaluru, L2}|[Python, SQL]      |
+------+--------+-------+---------------+-------------------+



In [9]:
emp_df.createOrReplaceTempView("emp_df")
dept_df.createOrReplaceTempView("dept_df")

In [10]:

spark.sql("""
          -- its lef_semi join

    select emp.*, dept.* from emp_df emp left join dept_df dept on emp.dept_id = dept.dept_id
          where emp.dept_id is not null and dept.dept_id is not null
""").show()

+------+--------+-------+---------------+-------------------+-------+----------------+--------------------+--------------------+
|emp_id|emp_name|dept_id|        details|             skills|dept_id|       dept_name|             hq_info|          req_skills|
+------+--------+-------+---------------+-------------------+-------+----------------+--------------------+--------------------+
|     1|    Amit|    101|     {Pune, L3}|[Spark, Scala, AWS]|    101|Data Engineering|     {Pune, Tower A}|[Spark, Scala, Ka...|
|     3|   Rohan|    101|   {Mumbai, L1}|      [Java, Spark]|    101|Data Engineering|     {Pune, Tower A}|[Spark, Scala, Ka...|
|     2|   Priya|    102|{Bengaluru, L2}|      [Python, SQL]|    102|       Analytics|{Bengaluru, Tower B}|[Python, SQL, Tab...|
+------+--------+-------+---------------+-------------------+-------+----------------+--------------------+--------------------+



In [11]:
# Left Anti Join ("left_anti", "anti")
# Works like WHERE NOT EXISTS. Returns only columns from emp_df 
# where dept_id does not match dept_df (including NULL keys).

emp_df.join(dept_df, emp_df.dept_id == dept_df.dept_id, "left_anti").show(truncate=False)


+------+--------+-------+-----------+------------------+
|emp_id|emp_name|dept_id|details    |skills            |
+------+--------+-------+-----------+------------------+
|4     |Sneha   |105    |{Pune, L2} |[PySpark, Airflow]|
|5     |Vikram  |NULL   |{Delhi, L4}|[Kafka, Scala]    |
+------+--------+-------+-----------+------------------+



In [12]:
## Natural Join
# Implicitly matches columns with the exact same name (dept_id) without writing a join condition, 
# and automatically deduplicates the dept_id column in the output.



# In PySpark DataFrame API, passing a list of column names acts like a Natural Join on those columns:
emp_df.join(dept_df, ["dept_id"], "inner").show(truncate=False)


spark.sql("SELECT * FROM emp_df NATURAL JOIN dept_df").show(truncate=False)

+-------+------+--------+---------------+-------------------+----------------+--------------------+----------------------+
|dept_id|emp_id|emp_name|details        |skills             |dept_name       |hq_info             |req_skills            |
+-------+------+--------+---------------+-------------------+----------------+--------------------+----------------------+
|101    |1     |Amit    |{Pune, L3}     |[Spark, Scala, AWS]|Data Engineering|{Pune, Tower A}     |[Spark, Scala, Kafka] |
|101    |3     |Rohan   |{Mumbai, L1}   |[Java, Spark]      |Data Engineering|{Pune, Tower A}     |[Spark, Scala, Kafka] |
|102    |2     |Priya   |{Bengaluru, L2}|[Python, SQL]      |Analytics       |{Bengaluru, Tower B}|[Python, SQL, Tableau]|
+-------+------+--------+---------------+-------------------+----------------+--------------------+----------------------+

+-------+------+--------+---------------+-------------------+----------------+--------------------+----------------------+
|dept_id|emp_id

In [13]:
# Cross (Cartesian) Join
# Matches every row in emp_df (5 rows) with every row in dept_df (4 rows) = 20 rows.

emp_df.crossJoin(dept_df).show(truncate=False)

+------+--------+-------+---------------+-------------------+-------+----------------+--------------------+------------------------+
|emp_id|emp_name|dept_id|details        |skills             |dept_id|dept_name       |hq_info             |req_skills              |
+------+--------+-------+---------------+-------------------+-------+----------------+--------------------+------------------------+
|1     |Amit    |101    |{Pune, L3}     |[Spark, Scala, AWS]|101    |Data Engineering|{Pune, Tower A}     |[Spark, Scala, Kafka]   |
|1     |Amit    |101    |{Pune, L3}     |[Spark, Scala, AWS]|102    |Analytics       |{Bengaluru, Tower B}|[Python, SQL, Tableau]  |
|1     |Amit    |101    |{Pune, L3}     |[Spark, Scala, AWS]|103    |Cloud Ops       |{Hyderabad, Tower C}|[AWS, Terraform, Docker]|
|1     |Amit    |101    |{Pune, L3}     |[Spark, Scala, AWS]|104    |Security        |{Mumbai, Tower A}   |[Linux, Networking]     |
|2     |Priya   |102    |{Bengaluru, L2}|[Python, SQL]      |101    |

In [14]:
# Join condition: Same dept_id AND employee works in the same city as the department HQ
struct_join_cond = (
    (emp_df.dept_id == dept_df.dept_id) & 
    (F.col("details.location") == F.col("hq_info.city"))
)

emp_df.join(dept_df, struct_join_cond, "inner").select(
    "emp_name", "dept_name", "details.location", "hq_info.city"
).show()

+--------+----------------+---------+---------+
|emp_name|       dept_name| location|     city|
+--------+----------------+---------+---------+
|    Amit|Data Engineering|     Pune|     Pune|
|   Priya|       Analytics|Bengaluru|Bengaluru|
+--------+----------------+---------+---------+



In [15]:
# Match employees to departments where skills[] and req_skills[] share at least 1 element
array_join_cond = F.arrays_overlap(emp_df.skills, dept_df.req_skills)

emp_df.join(dept_df, array_join_cond, "inner").select(
    "emp_name", "skills", "dept_name", "req_skills"
).show(truncate=False)

+--------+-------------------+----------------+------------------------+
|emp_name|skills             |dept_name       |req_skills              |
+--------+-------------------+----------------+------------------------+
|Amit    |[Spark, Scala, AWS]|Data Engineering|[Spark, Scala, Kafka]   |
|Amit    |[Spark, Scala, AWS]|Cloud Ops       |[AWS, Terraform, Docker]|
|Priya   |[Python, SQL]      |Analytics       |[Python, SQL, Tableau]  |
|Rohan   |[Java, Spark]      |Data Engineering|[Spark, Scala, Kafka]   |
|Vikram  |[Kafka, Scala]     |Data Engineering|[Spark, Scala, Kafka]   |
+--------+-------------------+----------------+------------------------+



In [16]:
# Example: Join using expr() or F.array_contains()
emp_df.join(
    dept_df,
    F.expr("array_contains(req_skills, skills[0])"), # Checks if employee's primary skill is in dept req_skills
    "inner"
).show(truncate=False)

+------+--------+-------+---------------+-------------------+-------+----------------+--------------------+----------------------+
|emp_id|emp_name|dept_id|details        |skills             |dept_id|dept_name       |hq_info             |req_skills            |
+------+--------+-------+---------------+-------------------+-------+----------------+--------------------+----------------------+
|1     |Amit    |101    |{Pune, L3}     |[Spark, Scala, AWS]|101    |Data Engineering|{Pune, Tower A}     |[Spark, Scala, Kafka] |
|2     |Priya   |102    |{Bengaluru, L2}|[Python, SQL]      |102    |Analytics       |{Bengaluru, Tower B}|[Python, SQL, Tableau]|
|5     |Vikram  |NULL   |{Delhi, L4}    |[Kafka, Scala]     |101    |Data Engineering|{Pune, Tower A}     |[Spark, Scala, Kafka] |
+------+--------+-------+---------------+-------------------+-------+----------------+--------------------+----------------------+



In [17]:
spark:SparkSession = spark